# 3. Fit material parameters at a fixed pose

**Goal:** learn the outer Torch/native derivative bridge before introducing equilibrium.
We hold the same two-tetrahedron mesh at a prescribed displacement, generate a synthetic
energy-gradient observation, and fit $p=(\lambda,\mu)$. No Newton solve occurs in this lesson.

The observation is a manufactured full internal-gradient vector, not an assumption
that all internal nodal forces can always be measured in an experiment.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
from litepgo import dof, energy, fem, solver
from litepgo.torch import TorchVolumetricMaterial, response
from common import make_geometry

torch.set_num_threads(1)
dtype = torch.float64

## Recreate the material and geometry

This is the same adapter as lesson 2, included so this lesson is independently runnable.

In [ ]:
class TorchNeoHookean(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        Fe = F @ torch.linalg.inv(Fp)
        lam, mu = theta.unbind(-1)
        log_J = torch.linalg.det(Fe).log()
        psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
        W = torch.linalg.det(Fp) * psi
        return torch.where((lam >= 0) & (mu > 0), W, torch.full_like(W, float('nan')))

X, topology, mapping, geometry, plastic = make_geometry()
key = (TorchNeoHookean, fem.TetP1Basis, fem.TetCentroidQuadrature)
truth = np.array([2., 3.])
u = (X @ np.array([[.08, .03, 0], [0, -.02, .02], [.01, 0, .04]]).T).ravel()

## Make parameters the input

`VolumetricFemForceMaterialResponse` fixes the pose and maps parameters to
$g(u,p)=\partial E/\partial u$. `ConstantDofMap(topology, 2)` shares one global
parameter pair across both elements. Local material-point derivatives are accumulated
into those same two global columns.

The response requests $\partial^2W/\partial F\partial\theta$ and assembles
$B=\partial g/\partial p$.

In [ ]:
parameter_map = dof.ConstantDofMap(topology, 2)
force = fem.VolumetricFemForceMaterialResponse[key](parameter_map, mapping, geometry, u, plastic)
target = torch.from_numpy(force.eval_value(truth).copy())
scale = target.square().mean().sqrt().item()
assert scale > 0
q = torch.tensor(np.log([1.2, 2.0]), dtype=dtype, requires_grad=True)

def loss(q):
    p = q.exp()
    predicted = response(force, p)
    return 0.5 * ((predicted - target) / scale).square().mean()

print('response input/output dimensions:', len(q), len(target))
print('initial loss:', loss(q).item())

## Understand the bridge and verify the gradient

The native response consumes the **explicit** tensor `p`. Its backward rule returns
$B^T\bar g$, the vector–Jacobian product. Torch then differentiates `p = exp(q)`.
This simple change of variables keeps both physical parameters positive.

The fixed `target`, `scale`, geometry, and pose do not change during optimization.
Compare the final loss gradient against finite differences of the whole loss.

In [ ]:
analytic = torch.autograd.grad(loss(q), q)[0]
h = 1e-4
numerical = torch.stack([(loss(q.detach() + d) - loss(q.detach() - d)) / (2 * h)
                         for d in h * torch.eye(len(q), dtype=dtype)])
torch.testing.assert_close(analytic, numerical, rtol=2e-4, atol=2e-6)
print('analytic gradient:', analytic.detach().numpy())
print('finite difference:', numerical.detach().numpy())

## Optimize

The closure may be called several times per L-BFGS step. Every call clears gradients,
recomputes the prediction and loss, and calls backward. We check loss reduction rather
than relying on the optimizer's printed status.

In [ ]:
history = []
optimizer = torch.optim.LBFGS([q], max_iter=25, tolerance_grad=1e-10,
                            tolerance_change=1e-14, line_search_fn='strong_wolfe')

def closure():
    optimizer.zero_grad()
    value = loss(q)
    value.backward()
    history.append(value.item())
    return value

before = loss(q).item()
optimizer.step(closure)
after = loss(q).item()
assert np.isfinite(after) and after < before * 1e-5
print('loss:', before, '->', after)
print('fitted material parameters:', q.detach().exp().numpy())
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.semilogy(np.maximum(history, 1e-18), marker='.')
ax.set(xlabel='L-BFGS closure evaluation (includes line-search trials)', ylabel='Loss')
ax.grid(alpha=0.25)
plt.show()
np.testing.assert_allclose(q.detach().exp().numpy(), truth, rtol=1e-3, atol=1e-3)

## Try it

Remove all shear from the prescribed pose, then examine `force.eval_jacobian(truth)`.
Are the two parameter columns still easy to distinguish? Small loss does not generally
prove unique parameter recovery.

**Checkpoint:** you can fit explicit material parameters through a native response.
Next: allow the pose itself to change when the parameters change.